# Etapa 4.1 - Persistencia con Redis
**TP IBD - Persistencia Políglota (NoSQL)**

Dominio: comercio minorista de suplementos deportivos.

Redis es un almacén de **estructuras de datos en memoria (RAM)**. Su fuerte no es "consultar"
como SQL o Mongo: es **acceder por una clave conocida con latencia sub-milisegundo**. Lo
usamos para los datos del dominio que se consultan muy seguido por su identificador y
representan el **estado actual** del sistema.

> Esta notebook resuelve el punto **4.1.1 - Modelo Clave-Valor y Hashes**. Las secciones
> 4.1.2 (listas como cola/historial) y 4.1.3 (TTL) se agregan más adelante.

## 1. Configuración del entorno

Se usa **Redis local con Docker** (imagen oficial):

```bash
docker run -d -p 6379:6379 --name redis-local-ibd redis:7.2-alpine
pip install redis
```

**Conexión:** `localhost:6379`. Usamos `decode_responses=True` para que Redis devuelva
strings de Python (no bytes), y así la salida es legible.

In [1]:
# Si hace falta instalar el cliente, descomentar:
# !pip install redis

import redis

r = redis.Redis(host="localhost", port=6379, decode_responses=True)

# Verificamos la conexión (lanza excepción si Redis no responde)
print("PING:", r.ping())
print("Redis version:", r.info("server")["redis_version"])

PING: True
Redis version: 7.2.14


### Limpieza idempotente
Para poder re-ejecutar la notebook sin residuos, borramos **solo las claves de este ejercicio**
(por prefijo, con `SCAN`). **No** usamos `FLUSHDB` para no afectar otros datos que pueda haber
en Redis.

In [2]:
PREFIJOS = ["stock:producto:", "stock:sucursal:", "producto:",
            "ventas:hoy:", "caja:", "pedidos:",
            "carrito:", "reserva:", "sesion:"]

borradas = 0
for pref in PREFIJOS:
    for k in r.scan_iter(match=pref + "*"):   # SCAN: itera sin bloquear (a diferencia de KEYS)
        r.delete(k)
        borradas += 1
print(f"Claves previas del ejercicio borradas: {borradas}")

Claves previas del ejercicio borradas: 27


## 2. Tipos de datos del dominio modelados (4.1.1)

Elegimos **tres** tipos de datos, todos consultados frecuentemente por su ID y que representan
el estado actual del negocio:

1. **Stock por producto y sucursal** → *hashes bidireccionales*. Es el caso central: queremos
   responder en **O(1)** "¿cuánto stock del producto P hay en la sucursal S?".
2. **Perfil de producto** → *hash* (`producto:<id>` con nombre, precio, marca).
3. **Contador de ventas del día** → *clave-valor simple* (`string`) con incremento atómico.

Definimos primero un pequeño catálogo del dominio (autocontenido y reproducible).

In [3]:
import random
random.seed(42)

# Catálogo de productos (subconjunto del dominio de la Etapa 1)
PRODUCTOS = [
    {"id": 1,  "nombre": "Whey Protein 100%",      "precio": 29000, "marca": "Ena Sport"},
    {"id": 2,  "nombre": "Isolate Whey Protein",   "precio": 42000, "marca": "Dymatize"},
    {"id": 3,  "nombre": "Creatina Monohidrato",   "precio": 22000, "marca": "Star Nutrition"},
    {"id": 4,  "nombre": "BCAA 2:1:1",             "precio": 15000, "marca": "Nutrilab"},
    {"id": 5,  "nombre": "Glutamina Micronizada",  "precio": 13500, "marca": "Pulver"},
    {"id": 6,  "nombre": "Quemador Termogenico",   "precio": 12500, "marca": "MuscleTech"},
    {"id": 7,  "nombre": "Pre-Entreno C4",         "precio": 33000, "marca": "BSN"},
    {"id": 8,  "nombre": "Multivitaminico Diario", "precio": 8000,  "marca": "Universal"},
    {"id": 9,  "nombre": "Omega 3 Ultra",          "precio": 9900,  "marca": "Optimum"},
    {"id": 10, "nombre": "Colageno Hidrolizado",   "precio": 18500, "marca": "Hoch Sport"},
]

SUCURSALES = {1: "Palermo", 2: "Belgrano", 3: "Centro", 4: "Caballito"}

print("Productos:", len(PRODUCTOS), "| Sucursales:", len(SUCURSALES))

Productos: 10 | Sucursales: 4


## 3. Tipo 1 — Stock bidireccional (caso central)

### El diseño
Un **hash** en Redis es una clave cuyo valor es un conjunto de pares `campo → valor`. Eso nos
permite resolver los dos niveles (producto y sucursal) en **un solo comando O(1)**.

Como necesitamos las **dos direcciones de búsqueda** rápido, mantenemos **dos hashes espejo**:

| Hash | Campo | Valor | Responde |
|------|-------|-------|----------|
| `stock:producto:<id>` | sucursal_id | cantidad | stock de un producto en cada sucursal |
| `stock:sucursal:<id>` | product_id  | cantidad | todos los productos de una sucursal |

Duplicar el dato es **a propósito**: en Redis se modela según el patrón de acceso. El precio a
pagar es que cada cambio de stock se escribe en los dos hashes (operaciones O(1), baratas).

- `HGET stock:producto:45 3` → un valor puntual, **O(1)** (no trae todo).
- `HGETALL stock:producto:45` → todos los campos, **O(N)** (N = cantidad de campos).

### Carga (`HSET`)
Generamos una matriz producto × sucursal y la escribimos en ambos hashes con un *pipeline*
(una sola ida y vuelta a Redis).

In [4]:
# Matriz de stock (producto, sucursal) -> cantidad
stock = {}
for p in PRODUCTOS:
    for sid in SUCURSALES:
        stock[(p["id"], sid)] = random.randint(20, 300)

# Carga en los DOS hashes espejo, con pipeline (menos round-trips)
pipe = r.pipeline()
for (pid, sid), cant in stock.items():
    pipe.hset(f"stock:producto:{pid}", sid, cant)   # vista por producto
    pipe.hset(f"stock:sucursal:{sid}", pid, cant)   # vista por sucursal (espejo)
pipe.execute()

print("Hashes de stock cargados.")
print("Ejemplo - stock:producto:1 =", r.hgetall("stock:producto:1"))

Hashes de stock cargados.
Ejemplo - stock:producto:1 = {'1': '77', '2': '32', '3': '160', '4': '145'}


### Consultas
La consulta clave del ejercicio: stock de **un** producto en **una** sucursal, en O(1).

In [5]:
# (1) Stock puntual: producto 1 en la sucursal 3 -> HGET key field, O(1)
pid, sid = 1, 3
cant = r.hget(f"stock:producto:{pid}", sid)
print(f"O(1)  Stock del producto {pid} en {SUCURSALES[sid]}: {cant} unidades")

# (2) Todas las sucursales de un producto -> HGETALL, O(N)
print(f"\nStock del producto {pid} en todas las sucursales:")
for s, c in r.hgetall(f"stock:producto:{pid}").items():
    print(f"   {SUCURSALES[int(s)]:<12} {c:>4}")

# (3) Direccion inversa: todos los productos de una sucursal -> HGETALL
nombre = {p["id"]: p["nombre"] for p in PRODUCTOS}
print(f"\nInventario de la sucursal {SUCURSALES[sid]}:")
for pr, c in r.hgetall(f"stock:sucursal:{sid}").items():
    print(f"   [{pr:>2}] {nombre[int(pr)]:<24} {c:>4}")

O(1)  Stock del producto 1 en Centro: 160 unidades

Stock del producto 1 en todas las sucursales:
   Palermo        77
   Belgrano       32
   Centro        160
   Caballito     145

Inventario de la sucursal Centro:
   [ 1] Whey Protein 100%         160
   [ 2] Isolate Whey Protein       72
   [ 3] Creatina Monohidrato       36
   [ 4] BCAA 2:1:1                139
   [ 5] Glutamina Micronizada     299
   [ 6] Quemador Termogenico      162
   [ 7] Pre-Entreno C4            194
   [ 8] Multivitaminico Diario    192
   [ 9] Omega 3 Ultra              69
   [10] Colageno Hidrolizado       42


### Actualización + verificación (requisito del punto)
Simulamos una **venta**: se venden 2 unidades del producto 1 en la sucursal 3. El stock baja
con `HINCRBY` (decremento **atómico**, O(1)), aplicado a **los dos hashes** dentro de una
transacción (`MULTI` vía pipeline) para que no se desfasen. Luego **verificamos** releyendo
ambas vistas: deben quedar con el mismo saldo.

In [6]:
pid, sid, vendidas = 1, 3, 2

antes = int(r.hget(f"stock:producto:{pid}", sid))

# Decremento atómico en AMBOS hashes, dentro de una transaccion
tx = r.pipeline(transaction=True)
tx.hincrby(f"stock:producto:{pid}", sid, -vendidas)
tx.hincrby(f"stock:sucursal:{sid}", pid, -vendidas)
tx.execute()

# Verificacion: releemos las dos vistas
desde_producto = int(r.hget(f"stock:producto:{pid}", sid))
desde_sucursal = int(r.hget(f"stock:sucursal:{sid}", pid))

print(f"Stock antes de la venta : {antes}")
print(f"Vendidas               : {vendidas}")
print(f"Stock (vista producto) : {desde_producto}")
print(f"Stock (vista sucursal) : {desde_sucursal}")
print("Consistencia entre las dos vistas:",
      "OK" if desde_producto == desde_sucursal == antes - vendidas else "DESFASADO")

Stock antes de la venta : 160
Vendidas               : 2
Stock (vista producto) : 158
Stock (vista sucursal) : 158
Consistencia entre las dos vistas: OK


## 4. Tipo 2 — Perfil de producto (hash)

Datos de un producto que se leen por ID (para mostrar una ficha). Un hash permite traer **un
campo** (`HGET`) o el documento completo (`HGETALL`) sin deserializar a mano.

In [7]:
# Carga de perfiles con HSET (mapping)
pipe = r.pipeline()
for p in PRODUCTOS:
    pipe.hset(f"producto:{p['id']}",
              mapping={"nombre": p["nombre"], "precio": p["precio"], "marca": p["marca"]})
pipe.execute()

# Consultas: un campo puntual y el perfil completo
print("Precio del producto 7 (HGET):", r.hget("producto:7", "precio"))
print("Perfil completo del producto 7 (HGETALL):", r.hgetall("producto:7"))

Precio del producto 7 (HGET): 33000
Perfil completo del producto 7 (HGETALL): {'nombre': 'Pre-Entreno C4', 'precio': '33000', 'marca': 'BSN'}


### Segunda actualización + verificación
Cambio de precio del producto 7 con `HSET` y verificación releyendo el campo.

In [8]:
print("Precio antes:", r.hget("producto:7", "precio"))
r.hset("producto:7", "precio", 35000)          # actualiza un solo campo
print("Precio despues:", r.hget("producto:7", "precio"))

Precio antes: 33000
Precio despues: 35000


## 5. Tipo 3 — Contador de ventas del día (clave-valor simple)

Un contador de ventas por sucursal: un `string` con incremento **atómico** (`INCR`). En
PostgreSQL esto requeriría un `UPDATE ... SET n = n+1` con bloqueo de fila; en Redis `INCR` es
atómico por diseño y O(1). También llevamos el total facturado con `INCRBYFLOAT`.

In [9]:
sid = 3
r.set(f"ventas:hoy:sucursal:{sid}", 0)     # inicializa el contador
r.set(f"caja:sucursal:{sid}", 0)

# Simulamos 5 ventas en la sucursal
for _ in range(5):
    r.incr(f"ventas:hoy:sucursal:{sid}")                 # +1 venta (atomico)
    r.incrbyfloat(f"caja:sucursal:{sid}", 29000.00)      # suma al total facturado

print(f"Ventas de hoy en {SUCURSALES[sid]}:", r.get(f"ventas:hoy:sucursal:{sid}"))
print(f"Facturado de hoy en {SUCURSALES[sid]}: $", r.get(f"caja:sucursal:{sid}"))

Ventas de hoy en Centro: 5
Facturado de hoy en Centro: $ 145000


## 6. ¿Por qué Redis y no PostgreSQL para estos datos?

- **Latencia sub-milisegundo por acceso directo a la clave.** El stock actual y el perfil de
  producto se consultan constantemente y siempre por su ID: es el patrón ideal para Redis
  (RAM + acceso O(1)), evitando ir a la base relacional en cada lectura.
- **Operaciones atómicas sin transacciones ni locks de fila.** `INCR` y `HINCRBY` incrementan
  de forma atómica por diseño; en PostgreSQL un contador muy golpeado genera contención de
  bloqueos sobre la misma fila.
- **Modelado según el patrón de acceso.** Mantener dos hashes espejo (por producto y por
  sucursal) da O(1) en **ambas** direcciones de consulta, algo que en SQL se resolvería con
  índices y `JOIN`s.

**Límite (importante):** Redis **no** reemplaza a PostgreSQL como **fuente de verdad** durable
ni para consultas analíticas o filtros arbitrarios (no hay `WHERE` ni `JOIN`). El esquema
correcto es **políglota**: PostgreSQL es la fuente de verdad transaccional y Redis es la capa
rápida de acceso al estado actual (caché/contadores), que se reconstruye desde la base si se
pierde.

## Nuevo requerimiento de negocio: ventas online con envío

El negocio incorpora ahora la **venta online con envío a domicilio**. A diferencia de la venta
en mostrador (que se entrega en el acto), un pedido online **no se entrega al instante**: queda
**pendiente de despacho** y atraviesa un **estado de envío** hasta llegar al cliente.

Por eso necesitamos **modelar el estado del envío** de cada pedido y gestionar los pedidos
pendientes como un flujo ordenado: se van **encolando** a medida que se concretan las ventas
online y se **despachan** en orden. La cola que modelamos a continuación representa, en sí
misma, el estado **"pendiente de envío"**: mientras un pedido está en la cola está esperando
ser despachado, y al salir de ella (despacho) pasa al estado **"en camino / despachado"**.

## 7. Lista como cola de pedidos a entregar (4.1.2)

Una **lista** de Redis preserva el **orden de inserción** y permite agregar/quitar en sus
extremos en **O(1)**. Es la estructura ideal para una **cola de trabajo**.

**Escenario:** los pedidos pendientes de entrega forman una cola **FIFO** (el primero que entra
es el primero que se despacha):

- **Al concretarse una venta** → `RPUSH`: el pedido entra por el **final** de la cola.
- **Al despachar** → `LPOP`: sale el pedido del **frente** (el más antiguo).

Así `RPUSH` + `LPOP` implementan FIFO, como una fila real (uno se forma atrás y se atiende
adelante). Cada pedido se guarda serializado como **JSON** (las listas almacenan strings).

> Clave usada: `pedidos:pendientes`. Para escalar, podría haber una cola por sucursal
> (`pedidos:pendientes:sucursal:<id>`) y que cada local despache la suya.

In [10]:
import json
COLA = "pedidos:pendientes"
r.delete(COLA)          # limpieza idempotente de la cola

random.seed(7)

def nuevo_pedido(venta_id):
    """Arma un pedido del dominio a partir de una venta."""
    sid = random.choice(list(SUCURSALES))
    items = random.sample(PRODUCTOS, random.randint(1, 4))
    return {
        "venta_id": venta_id,
        "sucursal": SUCURSALES[sid],
        "cliente_id": random.randint(1, 1500),
        "n_items": len(items),
        "total": sum(p["precio"] for p in items),
        "hora": f"{random.randint(9, 20):02d}:{random.randint(0, 59):02d}",
    }

def encolar_pedido(pedido):
    """Una venta nueva entra al FINAL de la cola (RPUSH)."""
    r.rpush(COLA, json.dumps(pedido))
    return pedido

def despachar_pedido():
    """Se despacha el del FRENTE = el mas antiguo (LPOP) -> FIFO."""
    raw = r.lpop(COLA)
    return json.loads(raw) if raw else None

# Carga inicial: 5 ventas entran a la cola
for vid in range(1050, 1055):
    p = encolar_pedido(nuevo_pedido(vid))
    print(f"Encolada venta {p['venta_id']} ({p['sucursal']}, ${p['total']})")

print("\nPedidos pendientes (LLEN):", r.llen(COLA))

Encolada venta 1050 (Centro, $62000)
Encolada venta 1051 (Centro, $9900)
Encolada venta 1052 (Caballito, $89000)
Encolada venta 1053 (Palermo, $47500)
Encolada venta 1054 (Palermo, $38900)

Pedidos pendientes (LLEN): 5


### Consultas sobre la cola
`LRANGE` para ver el contenido sin modificarlo, `LLEN` para el tamaño y `LINDEX 0` para espiar
el próximo a despachar.

In [11]:
# LRANGE 0 -1: toda la cola, del frente (mas antiguo) al final (mas nuevo)
print("Cola de pedidos pendientes (del mas antiguo al mas nuevo):")
for raw in r.lrange(COLA, 0, -1):
    p = json.loads(raw)
    print(f"   venta {p['venta_id']} | {p['sucursal']:<10} | {p['n_items']} items | ${p['total']}")

print("\nLLEN (cantidad pendiente):", r.llen(COLA))

# LINDEX 0: peek del proximo a despachar, SIN sacarlo de la cola
prox = json.loads(r.lindex(COLA, 0))
print(f"Proximo a despachar (LINDEX 0): venta {prox['venta_id']} - {prox['sucursal']}")

Cola de pedidos pendientes (del mas antiguo al mas nuevo):
   venta 1050 | Centro     | 2 items | $62000
   venta 1051 | Centro     | 1 items | $9900
   venta 1052 | Caballito  | 4 items | $89000
   venta 1053 | Palermo    | 2 items | $47500
   venta 1054 | Palermo    | 2 items | $38900

LLEN (cantidad pendiente): 5
Proximo a despachar (LINDEX 0): venta 1050 - Centro


### Gestión: despachar, encolar y cancelar
`LPOP` despacha el más antiguo (FIFO); `RPUSH` agrega una venta nueva; `RPOP` saca el último
ingresado (útil para cancelar el pedido recién cargado).

In [12]:
# Despachar el mas antiguo (LPOP) -> FIFO
desp = despachar_pedido()
print(f"Despachado: venta {desp['venta_id']} ({desp['sucursal']}). Quedan {r.llen(COLA)}")

# Llega una venta nueva -> se encola al final (RPUSH)
nuevo = encolar_pedido(nuevo_pedido(1055))
print(f"Encolada venta nueva {nuevo['venta_id']}. Quedan {r.llen(COLA)}")

# Cancelar el ULTIMO ingresado -> RPOP (extremo opuesto)
cancelado = json.loads(r.rpop(COLA))
print(f"Cancelada (RPOP) la ultima ingresada: venta {cancelado['venta_id']}. Quedan {r.llen(COLA)}")

Despachado: venta 1050 (Centro). Quedan 4
Encolada venta nueva 1055. Quedan 5
Cancelada (RPOP) la ultima ingresada: venta 1055. Quedan 4


### Simulación del flujo completo
Intercalamos ventas (entran con `RPUSH`) y despachos (salen con `LPOP`), mostrando cómo crece
y se vacía la cola en cada paso.

In [13]:
r.delete(COLA)       # arrancamos la simulacion con la cola vacia
random.seed(99)

print("=== Simulacion del flujo de pedidos ===\n")
eventos = ["venta", "venta", "despacho", "venta", "despacho",
           "despacho", "venta", "venta", "despacho"]
vid = 2000
for ev in eventos:
    if ev == "venta":
        p = encolar_pedido(nuevo_pedido(vid))
        print(f"[VENTA]    venta {p['venta_id']} encolada ({p['sucursal']:<10}) -> pendientes: {r.llen(COLA)}")
        vid += 1
    else:
        p = despachar_pedido()
        if p:
            print(f"[DESPACHO] sale  venta {p['venta_id']} ({p['sucursal']:<10}) -> pendientes: {r.llen(COLA)}")
        else:
            print("[DESPACHO] cola vacia, nada para despachar")

print(f"\nPedidos que quedaron sin despachar: {r.llen(COLA)}")

=== Simulacion del flujo de pedidos ===

[VENTA]    venta 2000 encolada (Caballito ) -> pendientes: 1
[VENTA]    venta 2001 encolada (Caballito ) -> pendientes: 2
[DESPACHO] sale  venta 2000 (Caballito ) -> pendientes: 1
[VENTA]    venta 2002 encolada (Belgrano  ) -> pendientes: 2
[DESPACHO] sale  venta 2001 (Caballito ) -> pendientes: 1
[DESPACHO] sale  venta 2002 (Belgrano  ) -> pendientes: 0
[VENTA]    venta 2003 encolada (Centro    ) -> pendientes: 1
[VENTA]    venta 2004 encolada (Centro    ) -> pendientes: 2
[DESPACHO] sale  venta 2003 (Centro    ) -> pendientes: 1

Pedidos que quedaron sin despachar: 1


## 8. Datos con tiempo de vida (TTL) (4.1.3)

Algunos datos del dominio tiene sentido que **expiren solos**: son temporales y mantenerlos
para siempre desperdiciaría memoria o, peor, dejaría información inconsistente. Redis permite
asociar un **TTL** (time-to-live) a cualquier clave; al vencer, Redis la borra automáticamente.

Elegimos **3 datos con TTL distinto y justificado**:

| Dato | Clave | TTL | Por qué ese tiempo |
|------|-------|-----|--------------------|
| **Carrito de compras** online | `carrito:cliente:<id>` | 30 min | Conserva la selección mientras el cliente compra, pero libera los carritos abandonados. Se **renueva** con cada acción. |
| **Reserva temporal de stock** | `reserva:stock:<prod>:<suc>` | 10 min | Reserva unidades mientras el cliente paga; si no concreta, el stock se libera para otros (evita sobreventa sin congelar inventario). |
| **Token de sesión** | `sesion:token:<token>` | 1 hora | Mantiene al cliente logueado en la tienda online; vence por seguridad ante inactividad. |

Los dos ejemplos nuevos (reserva de stock y token de sesión) se suman al del carrito.

In [14]:
# 1) Carrito de compras (hash) - expira por inactividad a los 30 min
r.delete("carrito:cliente:842")
r.hset("carrito:cliente:842", mapping={"1": 2, "12": 1})   # prod 1 x2, prod 12 x1
r.expire("carrito:cliente:842", 1800)                       # TTL 30 min

# 2) Reserva temporal de stock - se libera a los 10 min si no se paga
r.set("reserva:stock:1:3", 5, ex=600)                       # 5 unidades del prod 1 en suc 3

# 3) Token de sesion del cliente logueado - vence en 1 hora
r.set("sesion:token:abc123", 842, ex=3600)

print("3 claves creadas con TTL:")
print("  carrito:cliente:842   ->", r.ttl("carrito:cliente:842"), "s (30 min)")
print("  reserva:stock:1:3     ->", r.ttl("reserva:stock:1:3"), "s (10 min)")
print("  sesion:token:abc123   ->", r.ttl("sesion:token:abc123"), "s (1 hora)")

3 claves creadas con TTL:
  carrito:cliente:842   -> 1800 s (30 min)
  reserva:stock:1:3     -> 600 s (10 min)
  sesion:token:abc123   -> 3600 s (1 hora)


### Verificar el tiempo restante (`TTL` / `PTTL`)
`TTL` devuelve los segundos que le quedan a una clave; `PTTL`, los milisegundos. Convenciones:
`-1` = la clave existe pero **no** tiene expiración; `-2` = la clave **ya no existe**.

In [15]:
t_token = r.ttl("sesion:token:abc123")
t_res   = r.ttl("reserva:stock:1:3")

print(f"El token del usuario 842 sigue vigente por {t_token} segundos.")
print(f"La reserva del producto 1 en sucursal 3 se liberara en {t_res} segundos.")
print(f"PTTL del token (milisegundos): {r.pttl('sesion:token:abc123')}")

El token del usuario 842 sigue vigente por 3600 segundos.
La reserva del producto 1 en sucursal 3 se liberara en 600 segundos.
PTTL del token (milisegundos): 3599996


### Renovación y expiración automática
El carrito se **renueva** con cada acción del cliente (reiniciamos su TTL con `EXPIRE`).
Y demostramos la **expiración real**: creamos una reserva con TTL corto y verificamos que Redis
la elimina sola (en producción esta reserva sería de 10 min; acá usamos 2 s para poder verlo).

In [16]:
import time

# Renovacion: cada accion del cliente reinicia la cuenta regresiva del carrito
r.expire("carrito:cliente:842", 1800)
print("El cliente agrego un producto -> TTL del carrito reiniciado a",
      r.ttl("carrito:cliente:842"), "s\n")

# Expiracion real: reserva con TTL corto (2s) solo para la demostracion
r.set("reserva:stock:9:2", 3, ex=2)
print("Reserva temporal creada (TTL 2s). Vigente ahora?:", r.exists("reserva:stock:9:2") == 1)

time.sleep(3)   # esperamos a que venza

if r.exists("reserva:stock:9:2") == 0:
    print("La reserva del producto 9 expiro automaticamente: las 3 unidades se liberan.")
print("TTL de la clave expirada:", r.ttl("reserva:stock:9:2"), "(-2 = ya no existe)")

El cliente agrego un producto -> TTL del carrito reiniciado a 1800 s

Reserva temporal creada (TTL 2s). Vigente ahora?: True


La reserva del producto 9 expiro automaticamente: las 3 unidades se liberan.
TTL de la clave expirada: -2 (-2 = ya no existe)
